In [ ]:
import json
import os
import cv2
import re
import numpy as np
from pathlib import Path
from sklearn.cluster import KMeans

# --- 配置阈值 (通过直方图分析得出) ---
T_ROOT_AREA = 100.0   # 判定 Germinated
T_LEAF_AREA = 200.0   # 辅助判定 Shedding
T_DIST = 60.0        # 核心判定 Shedding

def natural_sort_key(s):
    """自然排序逻辑 (1, 2, ..., 10, 11)"""
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', str(s))]

def get_centroid(points):
    return np.mean(np.array(points), axis=0)

def get_area(points):
    x = [p[0] for p in points]; y = [p[1] for p in points]
    return 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

def main():
    # 1. 输入与路径准备
    raw_input = input("请输入包含原始JSON和图像的文件夹路径: ").strip()
    root_path = Path(raw_input.replace('"', '').replace("'", ""))
    
    if not root_path.exists():
        print("路径不存在！")
        return

    match_json_dir = root_path / "match_json"
    vis_dir = root_path / "vis"
    match_json_dir.mkdir(exist_ok=True)
    vis_dir.mkdir(exist_ok=True)

    # 自然排序确保时间轴正确
    json_files = sorted(list(root_path.glob("*.json")), key=natural_sort_key)
    if not json_files:
        print("未找到JSON文件。")
        return

    # 2. 全局槽位锚定 (KMeans)
    print("正在锚定36颗种子位置...")
    all_pts = []
    for j_p in json_files:
        with open(j_p, 'r', encoding='utf-8') as f:
            for s in json.load(f).get('shapes', []):
                all_pts.append(get_centroid(s['points']))
    
    kmeans = KMeans(n_clusters=36, n_init=20, random_state=42).fit(all_pts)
    centers = kmeans.cluster_centers_
    # 6x6 排序
    y_idx = np.argsort(centers[:, 1])
    sorted_centers = np.zeros_like(centers)
    for i in range(6):
        row = y_idx[i*6 : (i+1)*6]
        row = row[np.argsort(centers[row, 0])]
        sorted_centers[i*6 : (i+1)*6] = centers[row]

    # 3. 时间轴处理、状态锁与可视化绘制
    state_tracker = {i: 0 for i in range(36)}
    stage_map = {0: "Inert", 1: "Germinated", 2: "Shedding"}
    
    # 颜色定义 (BGR)
    part_colors = {"seed": (0, 255, 0), "root": (255, 120, 0), "leaf": (0, 0, 255)} 
    cond_colors = {0: (160, 160, 160), 1: (0, 255, 0), 2: (0, 0, 255)} 

    print("正在处理并可视化 (显示Condition与各器官关联ID)...")
    for j_path in json_files:
        with open(j_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        img_name = j_path.stem
        img_file = None
        for ext in ['.jpg', '.png', '.jpeg', '.JPG']:
            potential_img = root_path / f"{img_name}{ext}"
            if potential_img.exists():
                img_file = potential_img
                break
        
        img = cv2.imread(str(img_file)) if img_file else None
        frame_slots = {i: {'seed': None, 'root': None, 'leaf': None} for i in range(36)}
        
        # 4. 个体器官ID分配与状态判定准备
        for shape in data.get('shapes', []):
            label_lower = shape['label'].lower()
            centroid = get_centroid(shape['points'])
            dist = np.linalg.norm(sorted_centers - centroid, axis=1)
            slot_id = int(np.argmin(dist))
            shape['group_id'] = slot_id
            
            if 'seed' in label_lower: frame_slots[slot_id]['seed'] = shape['points']
            if 'root' in label_lower: frame_slots[slot_id]['root'] = shape['points']
            if 'leaf' in label_lower: frame_slots[slot_id]['leaf'] = shape['points']

        # 5. 状态判定（不绘制文字，先更新状态）
        for sid in range(36):
            parts = frame_slots[sid]
            current_instant_state = 0
            r_area = get_area(parts['root']) if parts['root'] else 0
            l_area = get_area(parts['leaf']) if parts['leaf'] else 0
            
            if parts['root'] and r_area > T_ROOT_AREA:
                current_instant_state = 1
            if parts['leaf']:
                if l_area > T_LEAF_AREA:
                    current_instant_state = 2
                elif parts['seed']:
                    d = np.linalg.norm(get_centroid(parts['leaf']) - get_centroid(parts['seed']))
                    if d > T_DIST:
                        current_instant_state = 2
            
            if current_instant_state > state_tracker[sid]:
                state_tracker[sid] = current_instant_state

        # ========== 6. 绘制部分（按优先级：先轮廓，后文字） ==========
        if img is not None:
            # 6.1 先绘制所有器官轮廓
            for shape in data['shapes']:
                label_lower = shape['label'].lower()
                pts = np.array(shape['points'], np.int32)
                line_color = (255, 255, 255)
                for key, color in part_colors.items():
                    if key in label_lower:
                        line_color = color
                        break
                cv2.polylines(img, [pts], True, line_color, 2)  # 线宽可根据需要调整

            # 6.2 再绘制ID文字（确保文字在轮廓上方）
            for shape in data['shapes']:
                gid = shape['group_id']
                shape['condition'] = stage_map[state_tracker[gid]]
                label_lower = shape['label'].lower()
                line_color = (255, 255, 255)
                for key, color in part_colors.items():
                    if key in label_lower:
                        line_color = color
                        break
                shape_centroid = get_centroid(shape['points']).astype(int)
                cv2.putText(img, f"id:{gid}", (shape_centroid[0]-12, shape_centroid[1]+5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, line_color, 1, cv2.LINE_AA)

            # 6.3 最后绘制槽位状态文字（在最上层）
            for sid in range(36):
                c_pt = sorted_centers[sid].astype(int)
                font_clr = cond_colors[state_tracker[sid]]
                final_cond_str = stage_map[state_tracker[sid]]
                text = f"Slot:{sid} {final_cond_str}"
                cv2.putText(img, text, (c_pt[0]-45, c_pt[1]-18),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, font_clr, 1, cv2.LINE_AA)

        # 7. 保存 match_json 和图像
        for shape in data['shapes']:
            if 'group_id' not in shape:
                shape['group_id'] = -1
            shape['condition'] = stage_map[state_tracker.get(shape.get('group_id', -1), 0)]

        with open(match_json_dir / j_path.name, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
            
        if img is not None:
            cv2.imwrite(str(vis_dir / img_file.name), img)

    print(f"处理完成！\n请在 vis 文件夹中检查多点关联效果。\n轮廓与ID颜色：Green=Seed, Blue=Root, Red=Leaf")

if __name__ == "__main__":
    main()

In [ ]:
import json
import re
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.cluster import KMeans

def natural_sort_key(s):
    """自然排序"""
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', str(s))]

def get_centroid(points):
    return np.mean(np.array(points), axis=0)

# ======================== 请修改这里的路径 ========================
# 替换为包含所有 JSON 文件和对应图像的文件夹路径
FOLDER_PATH = r"E:\guanxueying\graduation_project\thesis_fig_table\tobacco\predict"   # <--- 修改这里
# ================================================================

root_path = Path(FOLDER_PATH)
if not root_path.exists():
    raise FileNotFoundError(f"路径不存在: {root_path}")

# 创建输出子目录
output_dir = root_path / "kmeans_vis"
output_dir.mkdir(exist_ok=True, parents=True)

# 收集所有 JSON 文件（自然排序保持时间顺序）
json_files = sorted(list(root_path.glob("*.json")), key=natural_sort_key)
if not json_files:
    raise FileNotFoundError("未找到 JSON 文件")

# 1. 提取所有标注的质心点
all_pts = []
for j_p in json_files:
    with open(j_p, 'r', encoding='utf-8') as f:
        data = json.load(f)
        for shape in data.get('shapes', []):
            all_pts.append(get_centroid(shape['points']))
all_pts = np.array(all_pts)
print(f"提取到 {len(all_pts)} 个标注点")

# 2. KMeans 聚类 (36 个簇)
n_clusters = 36
kmeans = KMeans(n_clusters=n_clusters, n_init=20, random_state=42).fit(all_pts)
centers = kmeans.cluster_centers_
labels = kmeans.labels_

# 3. 对中心点进行 6×6 排序（按 y 优先，再按 x）
y_idx = np.argsort(centers[:, 1])
sorted_centers = np.zeros_like(centers)
for i in range(6):
    row = y_idx[i*6 : (i+1)*6]
    row = row[np.argsort(centers[row, 0])]
    sorted_centers[i*6 : (i+1)*6] = centers[row]

print("排序后的 36 个槽位中心点坐标 (Slot, X, Y):")
for i, (x, y) in enumerate(sorted_centers):
    print(f"Slot {i:2d}: ({x:.1f}, {y:.1f})")

# 4. 保存中心点到 CSV
df = pd.DataFrame(sorted_centers, columns=['X', 'Y'])
df.insert(0, 'Slot', range(36))
csv_path = output_dir / "sorted_centers.csv"
df.to_csv(csv_path, index=False)
print(f"中心点坐标已保存: {csv_path}")

# 5. Matplotlib 散点图（保存，不在 Notebook 中显示）
plt.figure(figsize=(12, 10))
scatter = plt.scatter(all_pts[:, 0], all_pts[:, 1], c=labels, s=10, cmap='tab20', alpha=0.6)
plt.scatter(centers[:, 0], centers[:, 1], c='red', marker='X', s=100, edgecolors='black', linewidth=0.5)
for i, (x, y) in enumerate(sorted_centers):
    plt.text(x, y, f'{i}', fontsize=16, ha='center', va='center',
             bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))
plt.title(f'KMeans Clustering of Seed Points (n_clusters={n_clusters})', fontsize=16)
plt.xlabel('X coordinate', fontsize=16)
plt.ylabel('Y coordinate', fontsize=16)
plt.gca().invert_yaxis()
plt.grid(alpha=0.3)
plt.tight_layout()
scatter_path = output_dir / "kmeans_scatter.png"
plt.savefig(scatter_path, dpi=300)
plt.close()
print(f"散点图已保存: {scatter_path}")

# 6. 在原图上叠加绘制（使用第一张图片作为底图）
first_img = None
for j_p in json_files:
    img_stem = j_p.stem
    for ext in ['.jpg', '.png', '.jpeg', '.JPG']:
        potential_img = root_path / f"{img_stem}{ext}"
        if potential_img.exists():
            first_img = cv2.imread(str(potential_img))
            break
    if first_img is not None:
        break

if first_img is not None:
    img_viz = first_img.copy()
    # 绘制所有点（绿色小圆）
    for (x, y) in all_pts:
        cv2.circle(img_viz, (int(x), int(y)), 2, (0, 255, 0), -1)
    # 绘制聚类中心（红色叉）
    for (x, y) in centers:
        cv2.drawMarker(img_viz, (int(x), int(y)), (0, 0, 255),
                       markerType=cv2.MARKER_CROSS, markerSize=10, thickness=2)
    # 绘制排序后的槽位编号
    for i, (x, y) in enumerate(sorted_centers):
        cv2.putText(img_viz, str(i), (int(x)-10, int(y)-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    overlay_path = output_dir / "kmeans_overlay.png"
    cv2.imwrite(str(overlay_path), img_viz)
    print(f"叠加图像已保存: {overlay_path}")
else:
    print("未找到对应的图像文件，无法生成叠加可视化。")

print(f"\n所有结果已保存到: {output_dir}")